# 03. 피처 엔지니어링 분석

파생 피처 분포, 상관관계, VIF, Mutual Information 분석

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

from src.data.loader import load_gaming_behavior
from src.features.engineer import engineer_gaming_behavior_features, get_feature_columns
from src.features.selector import calculate_correlation_matrix, calculate_vif

In [ ]:
df = load_gaming_behavior()
df = engineer_gaming_behavior_features(df)
print(f"Shape: {df.shape}")
df.head()

## 1. 파생 피처 분포 (이탈 vs 유지)

In [ ]:
derived_features = [
    "playtime_per_session", "weekly_activity_intensity",
    "session_engagement_score", "level_efficiency",
    "achievement_rate", "purchase_per_hour", "activity_score"
]

for feat in derived_features:
    fig = px.histogram(
        df, x=feat, color="is_churned", barmode="overlay",
        color_discrete_map={0: "#3498db", 1: "#e74c3c"},
        labels={"is_churned": "이탈 여부"},
        title=f"{feat} 분포 (이탈 vs 유지)",
        opacity=0.7
    )
    fig.show()

## 2. 상관관계 분석

In [ ]:
feature_cols = get_feature_columns("kaggle")
numeric_features = feature_cols["numeric"]

corr_matrix = df[numeric_features].corr()

fig = ff.create_annotated_heatmap(
    z=corr_matrix.values.round(2),
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale="RdBu_r",
    showscale=True
)
fig.update_layout(title="피처 상관관계 히트맵", width=900, height=700)
fig.show()

In [ ]:
to_drop = calculate_correlation_matrix(df[numeric_features], threshold=0.8)
print(f"상관관계 > 0.8인 제거 후보 피처: {to_drop}")

## 3. VIF (다중공선성) 분석

In [ ]:
vif_df = calculate_vif(df[numeric_features])
print("VIF 분석 결과:")
print(vif_df.to_string())
print(f"\nVIF > 10 피처: {vif_df[vif_df['VIF'] > 10]['feature'].tolist()}")

## 4. Mutual Information 기반 피처 중요도

In [ ]:
cat_features = feature_cols["categorical"]
df_enc = df.copy()
for col in cat_features:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

all_features = numeric_features + cat_features
X = df_enc[all_features]
y = df_enc["is_churned"]

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({"feature": all_features, "mi_score": mi_scores})
mi_df = mi_df.sort_values("mi_score", ascending=True)

fig = px.bar(mi_df, x="mi_score", y="feature", orientation="h",
             title="Mutual Information 피처 중요도",
             color="mi_score", color_continuous_scale="Viridis")
fig.update_layout(height=600)
fig.show()

## 5. 종합 비교

| 분석 방법 | 목적 | 결과 |
|-----------|------|------|
| 상관관계 | 다중공선성 피처 탐지 | 높은 상관 피처 쌍 식별 |
| VIF | 다중공선성 정량화 | VIF > 10 피처 확인 |
| Mutual Information | 타겟 변수와의 관련성 | 중요 피처 랭킹 |